[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 Hard: KV Cache Attention

Implement **multi-head attention with KV caching** for efficient autoregressive generation.

During LLM inference, recomputing all key/value projections at every step is wasteful.
A **KV cache** stores previously computed K and V tensors so only the new token(s) need projection.

### Signature
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — new tokens
        # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)
        # Returns: (output, (K_all, V_all))
```

### Requirements
- Inherit from `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` projections
- When `cache=None` (prefill): apply **causal mask**, return all K/V as cache
- When `cache` provided (decode): concat new K/V with cached, no causal mask needed for single-token decode
- Incremental decode must produce **identical** results to full forward pass

### Key Idea
```
Prefill:  [t0 t1 t2 t3] → full causal attention → cache = (K_{0:3}, V_{0:3})
Decode:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
Decode:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [6]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [7]:
import torch
import torch.nn as nn
import math

In [13]:
# ✏️ YOUR IMPLEMENTATION HERE

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        # pass  # Initialize W_q, W_k, W_v, W_o

    def forward(self, x, cache=None):
        # 1. Project Q, K, V from x
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # 2. Reshape to multi-head: (B, num_heads, S, d_k)
        B, S, D = x.shape
        q_cur = Q.view(B, S, self.num_heads, self.d_k)
        k_cur = K.view(B, S, self.num_heads, self.d_k)
        v_cur = V.view(B, S, self.num_heads, self.d_k)

        q_cur = q_cur.transpose(1, 2)
        k_cur = k_cur.transpose(1, 2)
        v_cur = v_cur.transpose(1, 2)


        # 3. If cache exists, concat new K/V with cached K/V
        # when cache is not None, it will arrive as a tuple
        mask = None

        if cache is not None:
            k_past, v_past = cache
            k_all = torch.cat([k_past, k_cur], dim=2)
            v_all = torch.cat([v_past, v_cur], dim=2)
            mask = False

        else:
            k_all, v_all = k_cur, v_cur
            mask = True

        # 4. Compute attention (causal mask needed during prefill)
        # we have q_cur, we have k_all and v_all
        scores = torch.matmul(q_cur, k_all.transpose(-1,-2)) / math.sqrt(self.d_k)

        if mask: # if mask is True that means we are still in prefill
          causal_mask = torch.triu(torch.ones(S, S), diagonal=1).bool()
          scores = scores.masked_fill(causal_mask, float('-inf'))

        weights = torch.softmax(scores, dim=-1)

        attn_output = torch.matmul(weights, v_all)

        attn_output = attn_output.transpose(1,2).contiguous().view(B, S, D)
        output = self.W_o(attn_output)

        return output, (k_all, v_all)

In [14]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

Full output shape: torch.Size([1, 6, 64])
Cache K shape: torch.Size([1, 4, 4, 16])
Match: True


In [15]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')


🧪 Testing: KV Cache Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (no cache) (6.8ms)
  ✅ [2/5] Cache structure (2.2ms)
  ✅ [3/5] Decode step appends to cache (2.1ms)
  ✅ [4/5] Incremental decode matches full forward (2.7ms)
  ✅ [5/5] Gradient flow (2.4ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (16.3ms total)
  Progress saved. Run status() to see your dashboard.

